## Data Ingestion

#### Langchain Documenttaion Structure 

In [1]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredFileLoader

def ingest_data_from_directory(directory_path: str):
    """
    Reads files from a directory and ingests them into LangChain documents.
    Supports TXT, PDF, and other unstructured formats.
    """
    documents = []

    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)

        if filename.endswith(".txt"):
            loader = TextLoader(file_path)
        elif filename.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        else:
            # fallback for other formats
            loader = UnstructuredFileLoader(file_path)

        docs = loader.load()
        documents.extend(docs)

    return documents



directory = "../data"
docs = ingest_data_from_directory(directory)
print(f"Ingested {len(docs)} chunks from {directory}")

d:\pyhton_projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ingested 1 chunks from ../data


In [2]:
docs

[Document(metadata={'producer': 'www.smallpdf.com', 'creator': 'www.smallpdf.com', 'creationdate': 'D:20250719132929', 'moddate': 'D:20250719132929', 'source': '../data\\Gautam_Rawat_resume_07_25.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='SkillsWORK EXPERIENCE\nTIAA GBS2021- Current\nAssociate Specialist\n❑ Leading and mentoring QA team members, promoting skill development\nand collaboration.\n❑ Managing defect triage meetings, analyzing defect trends, and ensures\ntimely resolution of critical issues.\n❑ Worked on classification model for participant contribution change.\n❑ Building tools and utility to be used across QA community.\nTIAA GBS2018-2020\nTest Analyst\n❑ Automated manual test cases for improved efficiency.\n❑ Managed and tracked test cases in JIRA.\n❑ Maintained test scripts and resolved execution issues.\n❑Prepared weekly team status reports.\nFIS2020-2021\nLead Engineer Testing\n❑ Ensured service quality and integration of Alipay payment flows.

In [3]:


from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    split_docs = text_splitter.split_documents(documents)
    return split_docs

In [4]:

split_docs = split_documents(docs)
split_docs

[Document(metadata={'producer': 'www.smallpdf.com', 'creator': 'www.smallpdf.com', 'creationdate': 'D:20250719132929', 'moddate': 'D:20250719132929', 'source': '../data\\Gautam_Rawat_resume_07_25.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='SkillsWORK EXPERIENCE\nTIAA GBS2021- Current\nAssociate Specialist\n❑ Leading and mentoring QA team members, promoting skill development\nand collaboration.\n❑ Managing defect triage meetings, analyzing defect trends, and ensures\ntimely resolution of critical issues.\n❑ Worked on classification model for participant contribution change.\n❑ Building tools and utility to be used across QA community.\nTIAA GBS2018-2020\nTest Analyst\n❑ Automated manual test cases for improved efficiency.\n❑ Managed and tracked test cases in JIRA.\n❑ Maintained test scripts and resolved execution issues.\n❑Prepared weekly team status reports.\nFIS2020-2021\nLead Engineer Testing\n❑ Ensured service quality and integration of Alipay payment flows.

In [5]:
len(docs), len(split_docs)

(1, 4)

In [8]:



from typing import List
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        self.model = None
        self.__load_model()
        
    def __load_model(self):
        self.model = SentenceTransformer(self.model_name)
        
    def generate_embedding(self, text:List[str]):
        print("Generating embeddings... for length:", len(text))
        embeddings =  self.model.encode(text,show_progress_bar=True)
        return embeddings

#initialize embedding manager and generate embeddings
embedding_manager = EmbeddingManager()
embedding_manager
       

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 223.79it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector Store

In [20]:


from typing import Any, List
import numpy as np
from sqlalchemy import desc


class VectorStoreManager:
    def __init__(self, collectiona_name="resumes_document",persist_directory="../data/vector_store"):
        import chromadb
        from chromadb.config import Settings

        self.collectiona_name = collectiona_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.collection = self.__initialize_store()
        
    def __initialize_store(self):
        import chromadb
        from chromadb.config import Settings

        self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )
        self.collection = self.client.get_or_create_collection(
            name=self.collectiona_name,
            metadata={"description": "Resume Embeddings for RAG"},
            )
        return self.collection
        
    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        #add documents to vector store
        ids = []
        metadatas=[]
        texts = []
        embeddings_list = []

        if len(documents) != len(embeddings):
            raise ValueError("Length of documents and embeddings must be equal.")

        for i, doc in enumerate(documents):
            ids.append(f"doc_{i}")
            texts.append(doc.page_content)
            embeddings_list.append(embeddings[i])

        self.collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings_list
        )
        
        
        print(f"Added {len(documents)} documents to the vector store.")
        

In [21]:

vecdotr_store_manager = VectorStoreManager()

In [22]:
vecdotr_store_manager

In [23]:
#generate embeddings for split documents
texts = [doc.page_content for doc in split_docs]
texts

['SkillsWORK EXPERIENCE\nTIAA GBS2021- Current\nAssociate Specialist\n❑ Leading and mentoring QA team members, promoting skill development\nand collaboration.\n❑ Managing defect triage meetings, analyzing defect trends, and ensures\ntimely resolution of critical issues.\n❑ Worked on classification model for participant contribution change.\n❑ Building tools and utility to be used across QA community.\nTIAA GBS2018-2020\nTest Analyst\n❑ Automated manual test cases for improved efficiency.\n❑ Managed and tracked test cases in JIRA.\n❑ Maintained test scripts and resolved execution issues.\n❑Prepared weekly team status reports.\nFIS2020-2021\nLead Engineer Testing\n❑ Ensured service quality and integration of Alipay payment flows.\n❑ Developed and maintained microservices test cases and scripts using\nJava, RestAssured, Mutation Testing, PACT Testing, Wiremock and\nSpringBoot.\n❑ Enhanced CI/CD pipelines for reliable production releases.\n❑Monitored and visualized logs with Splunk; set up

In [24]:
#generate embeddings for split documents
embeddings = embedding_manager.generate_embedding(texts)

#add to vector store
vecdotr_store_manager.add_documents(split_docs, embeddings)


Generating embeddings... for length: 4


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]

Added 4 documents to the vector store.


## Rag Retriever

In [27]:
from pydoc import doc
from turtle import distance


class Rag_Retriever:
    def __init__(self, vector_store_manager, embedding_manager):
        self.vector_store_manager = vector_store_manager
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5,score_threshold: float = 0.0):
        print(f"Retrieving top {top_k} documents for query: {query}")
        
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        # Perform similarity search in the vector store
        results = self.vector_store_manager.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        
        retrieved_docs = []
        
        if(results['documents'] and results['documents'][0]):
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results['distances'][0]
            ids = results['ids'][0]
        
            for i,(doc_id, doc, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                print(f"Document ID: {doc_id}")
                print(f"Distance: {distance}")
                print(f"Metadata: {metadata}")
                print(f"Content: {doc[:200]}...")  # Print first 200 characters
                print("-" * 50)
                
                similarity_score = 1 - distance  # Assuming distance is in [0, 2] for cosine similarity
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": doc,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

        return retrieved_docs  # Return the list of retrieved documents
    
Rag_Retriever= Rag_Retriever(vecdotr_store_manager, embedding_manager)

In [30]:
Rag_Retriever.retrieve("What is the languages?", top_k=3, score_threshold=0.1)

Retrieving top 3 documents for query: What is the languages?
Generating embeddings... for length: 1


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.74it/s]

Document ID: doc_1
Distance: 1.7779133319854736
Metadata: None
Content: Java, RestAssured, Mutation Testing, PACT Testing, Wiremock and
SpringBoot.
❑ Enhanced CI/CD pipelines for reliable production releases.
❑Monitored and visualized logs with Splunk; set up performance
...
--------------------------------------------------
Document ID: doc_2
Distance: 1.788450002670288
Metadata: None
Content: Framework to improve efficiency and consistency in quality assurance
processes.
❑ Maintained and updated test scripts, ensuring reliable and up-to-
date automated testing for ongoing service provision...
--------------------------------------------------
Document ID: doc_3
Distance: 1.8523420095443726
Metadata: None
Content: certification (AIF-C01)
Certified SAFe 5 Agilist
Education
Master of Computer Applications
Galgotias University (2014)
Bachelor of Computer
Applications H.N.B Gharwal
University (2011)
❑ Contribute...
--------------------------------------------------


[]